In [6]:
"""
Folio Score – Renewal & Refinance Extension
-------------------------------------------
This script trains a Folio Score model on historical data and applies it to new data,
handling missing values (e.g., '**') via imputation inside the pipeline.
"""

import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss
from sklearn.impute import SimpleImputer

# ---------------------------
# Data ingestion & cleaning
# ---------------------------

def read_safely(path: str) -> pd.DataFrame:
    encodings = ["utf-8", "utf-16", "cp1252", "latin-1"]
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Failed to read file {path} with known encodings.")

def validate_schema(df: pd.DataFrame) -> None:
    required = [
        "DFlag", "Original_amount", "Loan_term",
        "is_Loan_purpose_purc", "is_Loan_purpose_cash", "is_Loan_purpose_noca",
        "is_Prepayment_penalty_mortgage", "Mortgage_Insurance",
        "Credit_Score", "Debt_to_income",
        "OLoan_to_value", "CLoan_to_value",
        "Full_Appraisal", "Other_Appraisal",
        "Single_borrower"
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

def clean_numeric(df: pd.DataFrame, numeric_cols: list) -> pd.DataFrame:
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

# ---------------------------
# Folio Score model
# ---------------------------

@dataclass
class FolioScoreConfig:
    numeric_features: Tuple[str, ...] = (
        "Credit_Score", "Debt_to_income", "OLoan_to_value", "CLoan_to_value",
        "Original_amount", "Loan_term", "Mortgage_Insurance"
    )
    categorical_features: Tuple[str, ...] = (
        "is_Loan_purpose_purc", "is_Loan_purpose_cash", "is_Loan_purpose_noca",
        "Full_Appraisal", "Other_Appraisal", "Single_borrower"
    )
    test_size: float = 0.2
    random_state: int = 42
    solver: str = "lbfgs"
    max_iter: int = 1000

class FolioScoreModel:
    def __init__(self, config: Optional[FolioScoreConfig] = None):
        self.config = config or FolioScoreConfig()
        self.pipeline: Optional[Pipeline] = None
        self.calibrator: Optional[IsotonicRegression] = None

    def build_pipeline(self) -> Pipeline:
        numeric = list(self.config.numeric_features)
        categorical = list(self.config.categorical_features)

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),   # NEW: impute missing numeric
                    ("scaler", StandardScaler())
                ]), numeric),
                ("cat", Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),  # NEW: impute missing categorical
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]), categorical)
            ]
        )

        clf = LogisticRegression(
            solver=self.config.solver,
            max_iter=self.config.max_iter,
            class_weight="balanced"
        )

        return Pipeline(steps=[("pre", preprocessor), ("clf", clf)])

    def fit(self, df: pd.DataFrame) -> Dict[str, float]:
        X = df[list(self.config.numeric_features) + list(self.config.categorical_features)].copy()
        y = df["DFlag"].astype(int).values

        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=self.config.test_size, random_state=self.config.random_state, stratify=y
        )

        self.pipeline = self.build_pipeline()
        self.pipeline.fit(X_train, y_train)

        p_val_raw = self.pipeline.predict_proba(X_val)[:, 1]
        self.calibrator = IsotonicRegression(out_of_bounds="clip")
        self.calibrator.fit(p_val_raw, y_val)
        p_val_cal = self.calibrator.predict(p_val_raw)

        return {
            "brier_raw": brier_score_loss(y_val, p_val_raw),
            "brier_cal": brier_score_loss(y_val, p_val_cal)
        }

    def predict_default_prob(self, X: pd.DataFrame) -> np.ndarray:
        p_raw = self.pipeline.predict_proba(X)[:, 1]
        return self.calibrator.predict(p_raw)

    def folio_score(self, X: pd.DataFrame) -> np.ndarray:
        return 1.0 - self.predict_default_prob(X)

# ---------------------------
# Main execution
# ---------------------------

if __name__ == "__main__":
    # 1) Load training data
    train_path = "/tmp/Copy of fm36-1.csv"
    df_train = read_safely(train_path)
    validate_schema(df_train)

    # Clean numeric columns
    numeric_cols = list(FolioScoreConfig().numeric_features)
    df_train = clean_numeric(df_train, numeric_cols)

    # Drop rows missing target only
    df_train = df_train.dropna(subset=["DFlag"])

    # 2) Train model
    model = FolioScoreModel()
    metrics = model.fit(df_train)
    print("Calibration metrics:", metrics)

    # 3) Load new data for prediction
    predict_path = "/tmp/Copy of fm36-2.csv"
    df_new = read_safely(predict_path)
    validate_schema(df_new)
    df_new = clean_numeric(df_new, numeric_cols)

    # 4) Generate predictions
    df_new["Default_Prob"] = model.predict_default_prob(df_new)
    df_new["Folio_Score"] = model.folio_score(df_new)

    print("\nSample predictions from new dataset:")
    print(df_new[["Loanref", "Credit_Score", "Debt_to_income",
                  "OLoan_to_value", "Folio_Score"]].head())





Calibration metrics: {'brier_raw': np.float64(0.15426132608963716), 'brier_cal': np.float64(0.009056198022777958)}

Sample predictions from new dataset:
        Loanref  Credit_Score  Debt_to_income  OLoan_to_value  Folio_Score
0  F12Q10195288         782.0            22.0            60.0     1.000000
1  F12Q20126422         798.0            48.0            60.0     1.000000
2  F12Q30529249         800.0            45.0            36.0     1.000000
3  F12Q30403438         778.0            31.0            63.0     0.991979
4  F12Q40021654         755.0            30.0            71.0     0.986928
